In [1]:
import sys
import os
import pandas as pd
import numpy as np
import csv
import re
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import linearmodels as lm
from linearmodels import PanelOLS, RandomEffects
from scipy import stats
from linearmodels import RandomEffects
import statsmodels.api as sm
from linearmodels.panel.results import compare

In [2]:
final_df = pd.read_csv("data/prepared_data_for_regression.csv")
print(final_df.head(5))

      country  year  incidence  mortality  ln(GDP_per_capita)       bmi  \
0        fiji  2000  39.310026  33.722967            9.151239  0.303726   
1    cambodia  2000  11.807551  10.664151            7.561138  0.100004   
2    kiribati  2000  13.866566  12.601468            7.816584  0.307055   
3  kazakhstan  2000  35.284421  18.655098            9.467759  0.291659   
4     jamaica  2000  50.404926  25.599975            9.160964  0.240683   

     pop_65  urbanization_rate  fertility  female_labor_rate  \
0  3.437768             47.908      2.992             37.841   
1  2.816518             18.586      3.794             77.834   
2  3.413156             42.958      4.071                NaN   
3  6.693366             56.098      1.898             65.381   
4  6.060642             51.814      2.345             58.147   

   internet_penetration  health_exp  female_smoking  hosp_beds       MIR  
0              1.496850    3.424412            15.9       2.05  0.857872  
1             

In [3]:
initial_countries=final_df['country'].nunique()
print(initial_countries)

166


In [4]:
initial_years=final_df['year'].nunique()
print(initial_years)

24


# OECD  Countries:

In [5]:
oecd_countries = ['austria', 'australia', 'belgium', 'canada', 'chile', 
                  'colombia', 'czech', 'denmark', 'estonia', 'finland', 
                  'france', 'germany', 'greece', 'hungary', 'iceland', 'ireland', 
                  'israel', 'italy', 'japan', 'korea', 'latvia', 'lithuania', 'luxembourg', 
                  'mexico', 'netherlands', 'new zealand', 'norway', 'poland', 'portugal', 'slovakia', 
                  'slovenia', 'spain', 'sweden', 'switzerland', 'turkiye', 'united kingdom', 'united states', 'costa rica']
print(len(oecd_countries))

38


# Create Dummy Variables:

In [6]:
final_df['is_oecd'] = final_df['country'].isin(oecd_countries).astype(int)

# create interaction :

In [7]:
final_df['bmi_x_oecd'] = final_df['bmi']* final_df['is_oecd']
print(final_df['is_oecd'].value_counts())

0    2884
1     784
Name: is_oecd, dtype: int64


# Dummy Variables: 

In [8]:
final_df['is_oecd']= final_df['country'].str.lower().str.strip().isin(oecd_countries).astype(int)
print(final_df['is_oecd'].value_counts())

0    2884
1     784
Name: is_oecd, dtype: int64


1. Aging Society Dummy: Based on WHO standards, societies with >10% population over 65 are considered "ageing" or "aged". This captures potential non-linear increases in cancer incidence due to population structure.

In [9]:
final_df['dm_high_aging_society'] = (final_df['pop_65'] > 10).astype(int)
print(final_df['dm_high_aging_society'].value_counts())

0    2457
1    1211
Name: dm_high_aging_society, dtype: int64


2. High Life Expectancy Dummy threshold of 75 years represents advanced healthcare systems and higher probability of disease detection.

In [10]:
print(final_df[['year', 'country', 'is_oecd', 'dm_high_aging_society']].head(100))

    year       country  is_oecd  dm_high_aging_society
0   2000          fiji        0                      0
1   2000      cambodia        0                      0
2   2000      kiribati        0                      0
3   2000    kazakhstan        0                      0
4   2000       jamaica        0                      0
..   ...           ...      ...                    ...
95  2000       denmark        1                      1
96  2001          oman        0                      0
97  2000       lebanon        0                      0
98  2000       namibia        0                      0
99  2000  saudi arabia        0                      0

[100 rows x 4 columns]


# interaction of is_oecd variable and gdp

In [11]:
#final_df['Log GDP per capita']= np.log(final_df['gdp'])
final_df['Log GDP per capita_is_oecd']= final_df['ln(GDP_per_capita)']*final_df['is_oecd']
print(final_df['Log GDP per capita_is_oecd'].head(5))

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Log GDP per capita_is_oecd, dtype: float64


In [12]:
final_df['is_oecd'] = 0
final_df.loc[final_df['country'].isin(oecd_countries), 'is_oecd'] = 1

#interaction of is_oecd variable and gdp
#final_df['ln(GDP per capita)'] = np.log(final_df['gdp'])
final_df['log_gdp_is_oecd'] = final_df['ln(GDP_per_capita)'] * final_df['is_oecd']
# Filling NaNs in Interaction Term: Since 'NaN * 0 = NaN' in Python, rows where is_oecd is 0 but log_gdp is missing(NAN) 
# would incorrectly stay as NaN (and be counted in the results), then filled these with 0 to ensure only real OECD countries with data are counted.
final_df['log_gdp_is_oecd'] = final_df['log_gdp_is_oecd'].fillna(0)

# counting contries :
oecd_count = final_df[final_df['is_oecd'] == 1]['country'].nunique()
interaction_countries = final_df[final_df['log_gdp_is_oecd'] != 0]['country'].nunique()

In [13]:
print(final_df[['country','ln(GDP_per_capita)','is_oecd','log_gdp_is_oecd','dm_high_aging_society']].sample(20).round(2))

                   country  ln(GDP_per_capita)  is_oecd  log_gdp_is_oecd  \
3047                serbia                9.97        0             0.00   
2325             australia               10.88        1            10.88   
2905  syrian arab republic                8.58        0             0.00   
1221          south africa                9.46        0             0.00   
535              nicaragua                8.56        0             0.00   
396            philippines                8.56        0             0.00   
2821               iceland               10.97        1            10.97   
2521                guinea                8.11        0             0.00   
2390               belgium               10.96        1            10.96   
754                burundi                6.87        0             0.00   
18                 myanmar                7.22        0             0.00   
2958                 samoa                8.77        0             0.00   
140         

# delete percentage of rows with missing value(NA):

In [14]:
# report of missing values
missing_report = (final_df.isnull().sum() / len(final_df) * 100).round(2)
print(missing_report[missing_report > 0])

ln(GDP_per_capita)             1.09
female_labor_rate              2.48
internet_penetration           1.64
female_smoking                10.88
hosp_beds                      0.63
Log GDP per capita_is_oecd     1.09
dtype: float64


In [15]:
clean_df= final_df.dropna(subset=['incidence', 'ln(GDP_per_capita)', 'pop_65', 'urbanization_rate', 'fertility', 'female_labor_rate', 'log_gdp_is_oecd', 'internet_penetration', 'health_exp', 'female_smoking','hosp_beds'])
print(f" Missing values after deleting NA in clean_df: {clean_df.isnull().sum()}")

 Missing values after deleting NA in clean_df: country                       0
year                          0
incidence                     0
mortality                     0
ln(GDP_per_capita)            0
bmi                           0
pop_65                        0
urbanization_rate             0
fertility                     0
female_labor_rate             0
internet_penetration          0
health_exp                    0
female_smoking                0
hosp_beds                     0
MIR                           0
is_oecd                       0
bmi_x_oecd                    0
dm_high_aging_society         0
Log GDP per capita_is_oecd    0
log_gdp_is_oecd               0
dtype: int64


# Building stepwise regression

# Model 1: Adding Macro-Economic Variables

In [16]:
df_step= clean_df.set_index(['country', 'year'])
exog_1_fe= sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate']])
model_1_fe=PanelOLS(df_step['incidence'],exog_1_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered',
                                                                                                cluster_entity=True)
print(model_1_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.0066
Estimator:                   PanelOLS   R-squared (Between):              0.1529
No. Observations:                3124   R-squared (Within):               0.1110
Date:                Sun, Aug 30 2026   R-squared (Overall):              0.1503
Time:                        14:58:39   Log-likelihood                -1.005e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.8443
Entities:                         136   P-value                           0.0001
Avg Obs:                       22.971   Distribution:                  F(2,2963)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             0.5817
                            

# Model 2: Adding Demographic Variables:

In [17]:
exog_2_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate', 'log_gdp_is_oecd', 'pop_65','dm_high_aging_society','fertility']])
model_2_fe = PanelOLS(df_step['incidence'], exog_2_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered',
                                                                                                   cluster_entity=True)
print(model_2_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.0578
Estimator:                   PanelOLS   R-squared (Between):             -0.6239
No. Observations:                3124   R-squared (Within):               0.3306
Date:                Sun, Aug 30 2026   R-squared (Overall):             -0.6119
Time:                        14:58:39   Log-likelihood                   -9969.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      30.270
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(6,2959)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             2.3699
                            

# Model 3: Adding Lifestyle Variables:

In [18]:
exog_3_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65', 'dm_high_aging_society','log_gdp_is_oecd','fertility','bmi', 'female_smoking']])
model_3_fe=PanelOLS(df_step['incidence'], exog_3_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered', cluster_entity=True)
print(model_3_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.1210
Estimator:                   PanelOLS   R-squared (Between):              0.3290
No. Observations:                3124   R-squared (Within):               0.2289
Date:                Sun, Aug 30 2026   R-squared (Overall):              0.3209
Time:                        14:58:39   Log-likelihood                   -9861.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      50.868
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(8,2957)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             4.5243
                            

# Model 4: Adding Systemic and Digital variables:

In [19]:
exog_4_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65', 'dm_high_aging_society','log_gdp_is_oecd','fertility','bmi', 'female_smoking', 'female_labor_rate', 'internet_penetration']])
model_4_fe=PanelOLS(df_step['incidence'], exog_4_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                  cluster_entity=True)
print(model_4_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.1878
Estimator:                   PanelOLS   R-squared (Between):              0.4947
No. Observations:                3124   R-squared (Within):               0.5563
Date:                Sun, Aug 30 2026   R-squared (Overall):              0.4919
Time:                        14:58:40   Log-likelihood                   -9737.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      68.316
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(10,2955)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             7.8191
                            

# create female_labor_rate * is_oecd variable:

In [20]:
df_step['female_labor_rate * is_oecd']= df_step['female_labor_rate']* df_step['is_oecd']

# Model 5: Adding Interaction Terms Variables:

In [21]:
exog_5_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65', 'dm_high_aging_society','fertility','bmi', 'female_smoking', 'female_labor_rate', 'internet_penetration', 
                                   'log_gdp_is_oecd','female_labor_rate * is_oecd']])
model_5_fe=PanelOLS(df_step['incidence'], exog_5_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                  cluster_entity=True)
print(model_5_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.1972
Estimator:                   PanelOLS   R-squared (Between):              0.6430
No. Observations:                3124   R-squared (Within):               0.5488
Date:                Sun, Aug 30 2026   R-squared (Overall):              0.6380
Time:                        14:58:40   Log-likelihood                   -9719.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      65.979
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(11,2954)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             7.1956
                            

# Comparing results of all Fixed Effect models:

In [22]:
# Compare five models
Compare_results_fixed_Effect = compare(
    {'FE model 1': model_1_fe,
     'FE model 2': model_2_fe,
     'FE model 3': model_3_fe,
     'FE model 4': model_4_fe,
     'FE model 5': model_5_fe,},stars=True,)

# Convert the comparison summary to HTML and load it into a Pandas DataFrame
html_table = Compare_results_fixed_Effect.summary.as_html()
html_table = re.sub(r'(\.\d{3})\d+', r'\1', html_table)
df = pd.read_html(html_table)[0].fillna('')
df = df[~df.iloc[:, 0].isin(['R-squared', 'R-Squared (Between)', 'R-Squared (Overall)'])]
df.loc[df.iloc[:, 0].str.contains('Within', case=False), df.columns[0]] = ('R-squared')
df.iloc[0, 0] = ''
df.columns = ['', 'model 1', 'model 2', 'model 3', 'model 4', 'model 5']

# Find separator rows and clean their text
sep = df.index[
    df.astype(str).apply(lambda x: x.str.contains('===|---', na=False).any(), axis=1)].tolist()
df.loc[sep, :] = ''

# Set up the size of table & resulotion of png
fig, ax = plt.subplots(figsize=(11.5, len(df) * 0.32 + 0.3), dpi=300)
ax.axis('off')
# Create the table for showing results
t = ax.table(
    cellText=df.values,
    loc='center',
    cellLoc='center',
    colWidths=[0.22, 0.156, 0.156, 0.156, 0.156, 0.156],)
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.0, 1.5) 

# Style table's UI detail
for (r, c), cell in t.get_celld().items():
  cell.set_facecolor('white')
  top, bot, mid = (r == 0), (r == len(df) - 1), (r in sep)
  cell.set_edgecolor('black' if (top or bot or mid) else 'none')
  cell.visible_edges = 'TB' if top else ('B' if bot else ('T' if mid else ''))
  if top:
    cell.get_text().set_weight('bold')

# Add note for p-value enterpret
plt.figtext(0.08, 0.08,
    ('T-stats in parentheses\nNote: Standard errors in parentheses. *'' p<0.1, ** p<0.05, *** p<0.01'),
    fontsize=8,fontstyle='italic',fontfamily='serif',)

# Save the final Incidence table summary
plt.savefig('Incidence_comparison_Fixed_Effect.png', bbox_inches='tight',pad_inches=0.03, dpi=300,)
plt.close()

In [23]:
# Compare five models
Compare_results_fixed_Effect = compare(
    {'FE model 1': model_1_fe,
     'FE model 2': model_2_fe,
     'FE model 3': model_3_fe,
     'FE model 4': model_4_fe,
     'FE model 5': model_5_fe,},stars=True,)

# Convert the comparison summary to HTML and load it into a Pandas DataFrame
html_table = Compare_results_fixed_Effect.summary.as_html()
html_table = re.sub(r'(\.\d{3})\d+', r'\1', html_table)
df = pd.read_html(html_table)[0].fillna('')
df = df[~df.iloc[:, 0].isin(['R-squared', 'R-Squared (Between)', 'R-Squared (Overall)'])]
df.loc[df.iloc[:, 0].str.contains('Within', case=False), df.columns[0]] = 'R-squared'
df.iloc[0, 0] = ''
df.columns = ['', 'model 1', 'model 2', 'model 3', 'model 4', 'model 5']

df = df[~df.astype(str).apply(lambda x: x.str.contains('===|---', na=False).any(), axis=1)].reset_index(drop=True)
sep = df.index[(df.iloc[:, 0] == 'const') | (df.iloc[:, 0] == 'Effects')].tolist()

# Set up the size of table & resulotion of png
fig, ax = plt.subplots(figsize=(11.5, len(df) * 0.32 + 0.3), dpi=300)
ax.axis('off')
# Create the table for showing results
t = ax.table(
    cellText=df.values,
    loc='center',
    cellLoc='center',
    colWidths=[0.22, 0.156, 0.156, 0.156, 0.156, 0.156],)
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.0, 1.5)

# Style table's UI detail
last_row = len(df) - 1
for (r, c), cell in t.get_celld().items():
  cell.set_facecolor('white')
  top, bot, mid = (r == 0), (r == last_row), (r in sep)
  cell.set_edgecolor('black' if (top or bot or mid) else 'none')
  cell.visible_edges = 'TB' if top else ('B' if bot else ('T' if mid else ''))
  if top:
    cell.get_text().set_weight('bold')

# Add note for p-value enterpret
plt.figtext(0.08,0.08, ('T-stats in parentheses\nNote: Standard errors in parentheses. *'' p<0.1, ** p<0.05, *** p<0.01'),
    fontsize=8,fontstyle='italic',fontfamily='serif',)

# Save the final Incidence table summary
plt.savefig('Incidence_comparison_Fixed_Effect.png',bbox_inches='tight',pad_inches=0.03,dpi=300,)
plt.close()

# CSV File for ML model:

In [24]:
# creating CSV file for ML model:
df_step.to_csv('preparing_Incidence_data_for_ML.csv', index=False)

In [25]:
with open('incidence_ols_r2-within.txt', 'w') as f:
    f.write(str(model_5_fe.rsquared_within))